# Task 1: Tokenization and a FAQ Chatbot

## Part 1: Tokenization

Comparing whitespace splitting and a Unicode word regex on six inputs.

### Inputs and tokenizer functions

In [26]:
import re

texts = [
    "Hello, hey! Is this working?",
    "I don't know. I think it is.a",
    "Email me at ualam4050@gmail.com",
    "CS4944 is my AI course",
    "",
    "میں عزیر ہوں۔"
]


def whitespaceTokens(text):
    return text.split()


def regexTokens(text):
    return re.findall(r"\w+", text)

### Compare both methods

In [27]:
for text in texts:
    spaceList = whitespaceTokens(text)
    regexList = regexTokens(text)

    print("Original text:", repr(text))
    print("Whitespace tokens:", spaceList, "| Count:", len(spaceList))
    print("Regex tokens:", regexList, "| Count:", len(regexList))
    print()

Original text: 'Hello, hey! Is this working?'
Whitespace tokens: ['Hello,', 'hey!', 'Is', 'this', 'working?'] | Count: 5
Regex tokens: ['Hello', 'hey', 'Is', 'this', 'working'] | Count: 5

Original text: "I don't know. I think it is.a"
Whitespace tokens: ['I', "don't", 'know.', 'I', 'think', 'it', 'is.a'] | Count: 7
Regex tokens: ['I', 'don', 't', 'know', 'I', 'think', 'it', 'is', 'a'] | Count: 9

Original text: 'Email me at ualam4050@gmail.com'
Whitespace tokens: ['Email', 'me', 'at', 'ualam4050@gmail.com'] | Count: 4
Regex tokens: ['Email', 'me', 'at', 'ualam4050', 'gmail', 'com'] | Count: 6

Original text: 'CS4944 is my AI course'
Whitespace tokens: ['CS4944', 'is', 'my', 'AI', 'course'] | Count: 5
Regex tokens: ['CS4944', 'is', 'my', 'AI', 'course'] | Count: 5

Original text: ''
Whitespace tokens: [] | Count: 0
Regex tokens: [] | Count: 0

Original text: 'میں عزیر ہوں۔'
Whitespace tokens: ['میں', 'عزیر', 'ہوں۔'] | Count: 3
Regex tokens: ['میں', 'عزیر', 'ہوں'] | Count: 3



### Model tokenizer (if available)

This part tries a model tokenizer on two English examples, as the lab asks. If the library is not installed, the code prints a message and moves on.

In [28]:
print("--- Model tokenizer ---")

englishExamples = ["Tokenization is unbelievably useful.", "I don't know."]

try:
    import tiktoken
    encoder = tiktoken.get_encoding("cl100k_base")

    for text in englishExamples:
        ids = encoder.encode(text)
        pieces = []
        for tokenId in ids:
            pieces.append(encoder.decode([tokenId]))

        print("Original text:", repr(text))
        print("Token ids:", ids)
        print("Token pieces:", pieces)
        print("Count:", len(ids))
        print()
except Exception:
    print("Model tokenizer is not available on this computer.")

--- Model tokenizer ---
Model tokenizer is not available on this computer.


## Part 2: FAQ Chatbot (TF-IDF)

The chatbot has six FAQs, each with a source. It answers exact questions directly, asks for confirmation on similar questions, and says it has no answer when nothing matches.

### Chatbot code

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

faqs = [    
    {
        "question": "Hello",
        "answer": "Hey! What can i do for you?",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "Who are you?",
        "answer": "Hey! I'm a small FAQ chatbot made for a lab task. You ask me something and I try to find the best answer from my list.",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "What can you do?",
        "answer": "I can answer a few basic questions, like what tokenization is, what TF-IDF is, and how I find my answers. If your question is a bit different from mine, I'll just check with you first.",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "How do you find the answers?",
        "answer": "I compare the words in your question with the questions in my list using TF-IDF. Whichever one is closest to yours is the one I pick.",
        "source": "scikit-learn docs, Text feature extraction (tf-idf term weighting)"
    },    
    {
        "question": "Where do we submit this?",
        "answer": "To create Lab 6 folder in my gitbub and then commit it there",
        "source": "About this chatbot (written for this lab task)"
    },
    {
        "question": "Can you understand Urdu?",
        "answer": "Not really. My questions are in English, so I match English words best. Urdu words won't match anything in my list.",
        "source": "About this chatbot (written for this lab task)"
    }
]


threshold = 0.3


def cleanText(text):
    text = text.lower().strip()
    text = text.replace("?", "")
    return text


questions = []
for faq in faqs:
    questions.append(faq["question"])

vectorizer = TfidfVectorizer()
faqMatrix = vectorizer.fit_transform(questions)


def findExactFaq(userText):
    for i in range(len(faqs)):
        if cleanText(faqs[i]["question"]) == cleanText(userText):
            return i
    return -1


def findBestFaq(userText):
    userVector = vectorizer.transform([cleanText(userText)])
    scores = cosine_similarity(userVector, faqMatrix)[0]

    bestIndex = 0
    for i in range(len(scores)):
        if scores[i] > scores[bestIndex]:
            bestIndex = i
    return bestIndex, scores[bestIndex]


def showAnswer(faq):
    print("Bot:", faq["answer"])
    print("Source:", faq["source"])


def chat():
    print("FAQ Chatbot. Type quit to stop.")

    while True:
        userText = input("You: ")

        if cleanText(userText) == "quit":
            print("Bot: Goodbye!")
            break

        if cleanText(userText) == "":
            print("Bot: Please type a question.")
            continue

        exactIndex = findExactFaq(userText)
        if exactIndex != -1:
            showAnswer(faqs[exactIndex])
            continue

        bestIndex, score = findBestFaq(userText)

        if score >= threshold:
            print("Bot: Did you mean:", faqs[bestIndex]["question"], "(yes/no)")
            reply = input("You: ")

            if cleanText(reply) == "yes":
                showAnswer(faqs[bestIndex])
            else:
                print("Bot: Okay, please try asking in a different way.")
        else:
            print("Bot: Sorry, I do not have an answer for that.")

### Run the chatbot

Type these inputs one by one: `What can you do?` (exact answer), `tell me who you are` then `yes` (suggestion), press Enter on an empty line (empty input), `Tell me a joke` (unsupported), and finally `quit`.

In [30]:
chat()

FAQ Chatbot. Type quit to stop.
Bot: Hey! What can i do for you?
Source: About this chatbot (written for this lab task)
Bot: I can answer a few basic questions, like what tokenization is, what TF-IDF is, and how I find my answers. If your question is a bit different from mine, I'll just check with you first.
Source: About this chatbot (written for this lab task)
Bot: Goodbye!
